# VoleykoçAI: Türkçe MMLU Benchmark

Fine-tune edilmiş modeli `alibayram/yapay_zeka_turkce_mmlu` benchmark'ında ölçer ve base model ile karşılaştırır.

**Benchmark sıfırdan geliştirilmedi.** Hocanın ölçüm kodundaki (`olcum.py`) iki parça birebir korundu:
- cevap doğruluğunu kontrol eden `cevap_dogru_mu` fonksiyonu (harf eşleşmesi + anlamsal benzerlik yedeği),
- modele verilen prompt metni.

Tek fark: orijinal kod modeli **Ollama** üzerinden çağırıyor. Bizim çıktımız bir LoRA adaptörü, Ollama'ya doğrudan girmiyor; bu yüzden çıkarımı (inference) Unsloth/transformers ile yapıyorum. Ölçülen şey ve puanlama aynı.

Colab'da çalışır: `Runtime → Change runtime type → T4 GPU`.

| | |
|---|---|
| Benchmark | `alibayram/yapay_zeka_turkce_mmlu` (6200 soru, 62 bölüm, 5 şık) |
| Ölçülen 1 | `unsloth/Qwen3-4B-Instruct-2507` (base) |
| Ölçülen 2 | `berkcangumusisik/voleykoc-qwen3-4b-lora` (fine-tune) |
| Kıyas | liderlik tablosundan hazır skorlar |

## 1) Kurulum

In [ ]:
%pip install -q unsloth
%pip install -q --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git


In [ ]:
import torch

assert torch.cuda.is_available(), (
    "GPU yok. Runtime -> Change runtime type -> T4 GPU seçip yeniden başlat."
)
print(f"GPU: {torch.cuda.get_device_properties(0).name}")

## 2) Benchmark verisini yükle

olcum.py'nin okuduğu parquet'i okuyorum. Sütunlar: `soru`, `secenekler` (şık listesi), `cevap` (doğru şıkkın indeksi), `bolum`.

In [ ]:
import pandas as pd

MMLU_URL = "hf://datasets/alibayram/yapay_zeka_turkce_mmlu_model_cevaplari/data/train-00000-of-00001.parquet"
mmlu = pd.read_parquet(MMLU_URL, columns=["bolum", "soru", "cevap", "secenekler"])

# Tümünü ölçmek için None bırak. Hızlı deneme için bir sayı yaz (ör. 300).
LIMIT = None
if LIMIT:
    mmlu = mmlu.groupby("bolum", group_keys=False).apply(
        lambda g: g.head(max(1, LIMIT // mmlu["bolum"].nunique()))
    ).reset_index(drop=True)

print(f"{len(mmlu)} soru, {mmlu['bolum'].nunique()} bölüm")
print(mmlu.iloc[0]["soru"][:100])

## 3) olcum.py'den birebir alınan puanlama

Aşağıdaki `cevap_dogru_mu` fonksiyonu ve prompt metni hocanın `olcum.py` dosyasından değiştirilmeden kopyalandı.
Kaynak: https://huggingface.co/datasets/alibayram/yapay_zeka_turkce_mmlu_bolum_sonuclari/blob/main/olcum.py

In [ ]:
# Anlamsal benzerlik yedeği opsiyonel: Colab'da sürüm çakışması yaparsa
# harf-eşleşme tabanlı puanlamayla devam ediyoruz. olcum.py'nin ilk iki
# kontrolü (tam harf + ilk harf) birebir korunuyor.
try:
    from sentence_transformers import SentenceTransformer
    anlamsal_model = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
    print("Anlamsal yedek aktif.")
except Exception as e:
    anlamsal_model = None
    print(f"Anlamsal yedek devre dışı ({type(e).__name__}); harf eşleşmesiyle devam.")


def cevap_dogru_mu(dogru_cevap_index, verilen_cevap, secenekler):
    harfler = ['A', 'B', 'C', 'D', 'E']
    dogru_harf = harfler[dogru_cevap_index]
    verilen_cevap = verilen_cevap.upper().strip()

    if dogru_harf == verilen_cevap:
        return True
    elif len(verilen_cevap) > 1 and verilen_cevap[1] in [" ", ":", ")", "=", "-", "."]:
        return dogru_harf == verilen_cevap[0]
    elif anlamsal_model is not None:
        ec = anlamsal_model.encode([verilen_cevap])
        es = anlamsal_model.encode(secenekler)
        benzerlik = anlamsal_model.similarity(ec, es).tolist()[0]
        return benzerlik.index(max(benzerlik)) == dogru_cevap_index
    else:
        for ch in verilen_cevap:
            if ch in harfler:
                return ch == dogru_harf
        return False

In [ ]:
def prompt_kur(soru_metni, secenekler):
    """olcum.py ile birebir aynı prompt."""
    harfler = ['A', 'B', 'C', 'D', 'E']
    soru = soru_metni + "\n"
    for j in range(len(secenekler)):
        soru += harfler[j] + ": " + secenekler[j] + "\n"
    return (
        "Sana soru ve seçenekleri veriyorum. sadece hangi seçeneğin sorunun "
        "doğru cevabı olduğunu yaz. Örneğin 'A' veya 'B' gibi. Lütfen herhangi "
        "bir açıklama yapma!\nSoru: " + soru
    )

## 4) Bir modeli batch'li ölçen fonksiyon

6200 soruyu tek tek çalıştırmak yerine batch'liyorum; T4'te model başına süreyi dakikalara indiriyor. Cevap sadece bir harf olacağı için `max_new_tokens` küçük.

In [ ]:
import time
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048
BATCH = 16


def modeli_olc(model_adi, etiket):
    print(f"\n=== {etiket} yükleniyor: {model_adi} ===")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=model_adi,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)
    tokenizer.padding_side = "left"  # decoder-only üretimde sol padding gerekir

    promptlar = [
        prompt_kur(mmlu.iloc[i]["soru"], list(mmlu.iloc[i]["secenekler"]))
        for i in range(len(mmlu))
    ]

    cevaplar = []
    basla = time.time()
    for b in range(0, len(promptlar), BATCH):
        grup = promptlar[b:b + BATCH]
        metinler = [
            tokenizer.apply_chat_template(
                [{"role": "user", "content": p}],
                tokenize=False, add_generation_prompt=True,
            )
            for p in grup
        ]
        girdi = tokenizer(metinler, return_tensors="pt", padding=True).to("cuda")
        with torch.no_grad():
            cikti = model.generate(
                **girdi, max_new_tokens=8, do_sample=False,
                repetition_penalty=1.3,
                eos_token_id=tokenizer.convert_tokens_to_ids("<|im_end|>"),
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )
        yeni = cikti[:, girdi["input_ids"].shape[1]:]
        cevaplar.extend(tokenizer.batch_decode(yeni, skip_special_tokens=True))

        gecen = time.time() - basla
        n = min(b + BATCH, len(promptlar))
        print(f"\r  {n}/{len(promptlar)}  ({gecen:.0f} sn)", end="")

    # Puanlama: olcum.py mantığı
    bolum_dogru, bolum_toplam = {}, {}
    toplam_dogru = 0
    for i in range(len(mmlu)):
        bolum = mmlu.iloc[i]["bolum"]
        bolum_toplam[bolum] = bolum_toplam.get(bolum, 0) + 1
        dogru = cevap_dogru_mu(
            int(mmlu.iloc[i]["cevap"]), cevaplar[i], list(mmlu.iloc[i]["secenekler"])
        )
        if dogru:
            toplam_dogru += 1
            bolum_dogru[bolum] = bolum_dogru.get(bolum, 0) + 1

    basari = round(toplam_dogru / len(mmlu) * 100, 2)
    print(f"\n  {etiket}: {toplam_dogru}/{len(mmlu)} = %{basari}  "
          f"({time.time() - basla:.0f} sn)")

    del model
    torch.cuda.empty_cache()

    return {
        "etiket": etiket, "model": model_adi, "basari": basari,
        "dogru": toplam_dogru, "toplam": len(mmlu),
        "bolum_dogru": bolum_dogru, "bolum_toplam": bolum_toplam,
        "cevaplar": cevaplar,
    }

## 5) Base ve fine-tune modellerini ölç

İki model art arda yükleniyor. Base yaklaşık 15-20 dk, fine-tune benzer. Toplam ~30-40 dk.

In [ ]:
base = modeli_olc("unsloth/Qwen3-4B-Instruct-2507", "Base Qwen3-4B")

In [ ]:
ft = modeli_olc("berkcangumusisik/voleykoc-qwen3-4b-lora", "VoleykoçAI (fine-tune)")

## 6) Sonuçlar ve model kartı için tablo

Liderlik tablosundan birkaç referans skoru da ekliyorum (bunları kendim çalıştırmadım, hocanın yayımladığı değerler).

In [ ]:
# Liderlik tablosundan referans skorlar (yayımlanmış değerler)
lider = pd.read_parquet(
    "hf://datasets/alibayram/yapay_zeka_turkce_mmlu_liderlik_tablosu/data/train-00000-of-00001.parquet"
)
referanslar = ["qwen3:14b", "gemma2:latest", "qwen2.5:latest", "llama3.1:latest"]
ref = lider[lider["model"].isin(referanslar)][["model", "parameter_size", "basari"]]
print(ref.to_string(index=False))

In [ ]:
fark = round(ft["basari"] - base["basari"], 2)

print("## Türkçe MMLU Sonuçları\n")
print(f"Benchmark: `alibayram/yapay_zeka_turkce_mmlu`, {base['toplam']} soru, 62 bölüm.\n")
print("| Model | Parametre | Başarı |")
print("|---|---|---:|")
print(f"| **VoleykoçAI (fine-tune)** | 4B (LoRA) | **%{ft['basari']}** |")
print(f"| Base Qwen3-4B-Instruct-2507 | 4B | %{base['basari']} |")
for _, r in ref.iterrows():
    print(f"| {r['model']} (liderlik) | {r['parameter_size']} | %{r['basari']} |")
print(f"\nFine-tune farkı: **{fark:+} puan**.")

In [ ]:
# Bölüm bazında en çok değişen 10 kategori
satirlar = []
for bolum in base["bolum_toplam"]:
    t = base["bolum_toplam"][bolum]
    b = base["bolum_dogru"].get(bolum, 0) / t * 100
    f = ft["bolum_dogru"].get(bolum, 0) / t * 100
    satirlar.append((bolum, t, round(b, 1), round(f, 1), round(f - b, 1)))

satirlar.sort(key=lambda x: abs(x[4]), reverse=True)
print("| Bölüm | Soru | Base | Fine-tune | Fark |")
print("|---|---:|---:|---:|---:|")
for bolum, t, b, f, d in satirlar[:10]:
    print(f"| {bolum} | {t} | %{b} | %{f} | {d:+} |")

In [ ]:
# Sonucu dosyaya yaz: indirip reports/ altına koy
import json

ozet = {
    "benchmark": "alibayram/yapay_zeka_turkce_mmlu",
    "soru_sayisi": base["toplam"],
    "base": {"model": base["model"], "basari": base["basari"], "dogru": base["dogru"]},
    "finetune": {"model": ft["model"], "basari": ft["basari"], "dogru": ft["dogru"]},
    "fark": fark,
    "bolum_karsilastirma": [
        {"bolum": s[0], "soru": s[1], "base": s[2], "finetune": s[3], "fark": s[4]}
        for s in satirlar
    ],
}
with open("mmlu_sonuclari.json", "w", encoding="utf-8") as fh:
    json.dump(ozet, fh, ensure_ascii=False, indent=2)
print("mmlu_sonuclari.json yazıldı. İndirip reports/ altına koy.")